# ReefScan-ID: Coralscapes EDA

Exploration of the Coralscapes dataset (module A, segmentation) before any class mapping or preprocessing is designed. Run in Google Colab with a T4 GPU runtime.

### Setup: Colab runtime and Google Drive

In [2]:
import os
import torch
from google.colab import drive

drive.mount("/content/drive")

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("GPU: {}".format(gpu_name))

base_dir = "/content/drive/MyDrive/ReefScan-ID"
print("Drive folder found: {}".format(os.path.isdir(base_dir)))
if os.path.isdir(base_dir):
    print("Contents: {}".format(sorted(os.listdir(base_dir))))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
Drive folder found: True
Contents: ['01_eda_coralscapes.ipynb', 'Checkpoints', 'Processed']


### Setup: libraries

In [3]:
!pip install -q datasets

import datasets
print("datasets version: {}".format(datasets.__version__))

datasets version: 4.8.5


## Section 1: Splits and image sizes

Question: do the number of images per split and the image and mask sizes match what the dataset card states (train 1,517, validation 166, test 392; images of 1024 by 2048 pixels)?

### Load the dataset

In [4]:
from datasets import load_dataset

ds = load_dataset("EPFL-ECEO/coralscapes")
print(ds)

README.md:   0%|          | 0.00/13.1k [00:00<?, ?B/s]

data/train-00000-of-00009.parquet: reconstructing file:   0%|          |  0.00B /  472MB            

data/train-00000-of-00009.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00009.parquet: reconstructing file:   0%|          |  0.00B /  448MB            

data/train-00001-of-00009.parquet: downloading bytes:           |  0.00B            

...

data/test-00002-of-00003.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1517 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/166 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/392 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 1517
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 166
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 392
    })
})


### Images per split

In [5]:
for split in ds:
    print("{}: {} images".format(split, ds[split].num_rows))

train: 1517 images
validation: 166 images
test: 392 images


### Image and mask sizes

In [6]:
import io
from collections import Counter
from PIL import Image
from datasets import Image as HFImage

# Read only the file headers, without decoding every full image
raw = ds.cast_column("image", HFImage(decode=False)).cast_column("label", HFImage(decode=False))

for split in raw:
    image_sizes = Counter()
    mask_sizes = Counter()
    for row in raw[split]:
        img = Image.open(io.BytesIO(row["image"]["bytes"]))
        mask = Image.open(io.BytesIO(row["label"]["bytes"]))
        # PIL gives (width, height), so swap to height x width
        image_sizes["{}x{} {}".format(img.size[1], img.size[0], img.mode)] += 1
        mask_sizes["{}x{} {}".format(mask.size[1], mask.size[0], mask.mode)] += 1
    print("{} images (height x width, mode): {}".format(split, dict(image_sizes)))
    print("{} masks (height x width, mode): {}".format(split, dict(mask_sizes)))

train images (height x width, mode): {'1024x2048 RGB': 1517}
train masks (height x width, mode): {'1024x2048 L': 1517}
validation images (height x width, mode): {'1024x2048 RGB': 166}
validation masks (height x width, mode): {'1024x2048 L': 166}
test images (height x width, mode): {'1024x2048 RGB': 392}
test masks (height x width, mode): {'1024x2048 L': 392}


**Observation**: